<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/2_basic_project_examples/7_natural_language_to_sql/7_natural_language_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ask Your Database in Plain English

Type a question like a normal person; the app **writes** the SQL, **runs** it against a real database, and **hands back** the answer. Same basic call, aimed at code instead of prose.

## Setup

Install the library and connect the client to your API key.

In [1]:
%pip install -q openai==2.28.0 pandas==2.2.2

import os
from openai import OpenAI

# Key from Colab secrets, an environment variable, or a prompt, in that order.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
TEXT_MODEL = "gpt-5-nano"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/robertbarcik/git-repos/ADK-tutorial/.venv/bin/python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## The database

A tiny IT helpdesk database: two tables, a few dozen rows, loaded straight from the CSVs in this folder into SQLite.

In [2]:
import sqlite3
import pandas as pd

tickets = pd.read_csv("tickets.csv")
technicians = pd.read_csv("technicians.csv")

db = sqlite3.connect(":memory:")
tickets.to_sql("tickets", db, index=False)
technicians.to_sql("technicians", db, index=False)

tickets.head(3)

,ticket_id,title,category,priority,status,days_ago_created,days_ago_resolved,assigned_to,customer_company
0,T001,Server outage - production environment down,Network Problem,Critical,In Progress,1,NaN,Sarah Johnson,SecureBank
1,T002,Cannot access shared drive - entire department...,Access Request,Critical,Open,2,NaN,Mike Chen,TechCorp
2,T003,Email server not responding,Email Issue,Critical,Resolved,3,2.0,Sarah Johnson,DataSystems


## The question you ask

Change this, re-run, get a new query and a new answer.

In [3]:
question = "Which technician has resolved the most tickets?"

## The one call

This is the whole engine: hand the model the schema and the question, get SQL back.

In [4]:
schema = (
    f"tickets({', '.join(tickets.columns)})\n"
    f"  example rows: {tickets.head(2).to_dict('records')}\n"
    f"technicians({', '.join(technicians.columns)})\n"
    f"  example rows: {technicians.head(2).to_dict('records')}"
)

sql = client.responses.create(
    model=TEXT_MODEL,
    input=[
        {"role": "developer", "content":
            f"Write one SQLite SELECT query that answers the question, using only these tables:\n{schema}\n"
            "Return only the SQL, no explanation, no markdown."},
        {"role": "user", "content": question},
    ],
).output_text.strip().strip("`").removeprefix("sql").strip()

print(sql)

SELECT t.name AS technician_name, COUNT(*) AS resolved_tickets
FROM technicians t
JOIN tickets tk ON tk.assigned_to = t.name
WHERE tk.days_ago_resolved IS NOT NULL
GROUP BY t.name
ORDER BY resolved_tickets DESC, technician_name ASC
LIMIT 1;


## Run it

Execute the generated SQL and show the answer.

In [5]:
# Never run unchecked model output against a real database - this is the minimum guardrail.
assert sql.upper().startswith("SELECT"), "Refusing to run a non-SELECT query"

pd.read_sql(sql, db)

,technician_name,resolved_tickets
0,Alex Rodriguez,3


## Take it further

One question already gets you SQL, a real query, and an answer. A few directions to riff on (good whiteboard fodder, and the API call barely changes):

- **Explain in English.** Send the result back to the model and ask it to summarize the answer in one sentence.
- **Stronger guardrails.** Reject forbidden keywords (`DROP`, `DELETE`, `UPDATE`) before running anything in production.
- **Your own data.** Point `tickets`/`technicians` at your own CSVs, or swap in a real database connection.

Pick one, sketch the pseudocode, and notice how little new code it needs.

### Your turn

Change `question` above and re-run the last two cells for a new answer.